In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: load-locked-master
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
MASTER = ROOT / "results/comprehensive_latest_48_models/all_48_models_master.csv"
master = pd.read_csv(MASTER)

pd.DataFrame({
    "quantity": ["rows", "architecture families", "unique masks",
                 "primary seed", "reported endpoints"],
    "value": [len(master), master.family.nunique(),
              master.factorial_mask.nunique(), 42, master.shape[1] - 7]
})

,quantity,value
0,rows,48
1,architecture families,3
2,unique masks,16
3,primary seed,42
4,reported endpoints,85


In [3]:
#| label: full-versus-base
#| tbl-cap: Full-composite minus MSE-only descriptive changes.
anchors = master[master.factorial_mask.isin([1000, 1111])].copy()
wide = anchors.pivot(index="family", columns="factorial_mask")
delta = pd.DataFrame({
    "family": wide.index,
    "delta_ptbxl_pearson": (
        wide["ptbxl_missing_pearson"][1111] -
        wide["ptbxl_missing_pearson"][1000]
    ).values,
    "delta_qrs_corr": (
        wide["ptbxl_qrs_correlation"][1111] -
        wide["ptbxl_qrs_correlation"][1000]
    ).values,
    "delta_st_corr": (
        wide["ptbxl_st_correlation"][1111] -
        wide["ptbxl_st_correlation"][1000]
    ).values,
    "delta_echonext_shd_auroc": (
        wide["echonext_shd_macro_auroc"][1111] -
        wide["echonext_shd_macro_auroc"][1000]
    ).values,
})
delta

,family,delta_ptbxl_pearson,delta_qrs_corr,delta_st_corr,delta_echonext_shd_auroc
0,ecg_aim,0.008399,0.007616,0.020798,-0.001221
1,multiscale_vae,0.009587,0.008726,0.016605,-0.009528
2,unet,0.037842,0.036469,0.048535,-0.021452


In [4]:
#| label: familywise-tests
#| tbl-cap: Prespecified patient-cluster BCa endpoint tests.
tests = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/raw/factorial_v4_2x4/"
           "familywise_endpoint_tests.csv"
)
tests[["family", "endpoint", "estimate", "ci_low", "ci_high",
       "p_value", "alpha", "reject_null"]]

,family,endpoint,estimate,ci_low,ci_high,p_value,alpha,reject_null
0,unet,QRS,0.011297,0.009998,0.012671,0.000500,0.0167,True
1,unet,ST,0.020933,0.019444,0.022527,0.000500,0.0167,True
2,unet,diagnostic_utility,0.003921,-0.000575,0.008677,0.100450,0.0167,False
3,msvae,QRS,0.004518,0.003957,0.005150,0.000500,0.0167,True
4,msvae,ST,0.009247,0.008235,0.010247,0.000500,0.0167,True
5,msvae,diagnostic_utility,0.000206,-0.002271,0.002727,0.876562,0.0167,False
6,ecgaim,QRS,0.008231,0.007518,0.008954,0.000500,0.0167,True
7,ecgaim,ST,0.013526,0.012348,0.014856,0.000500,0.0167,True
8,ecgaim,diagnostic_utility,0.000109,-0.001827,0.002355,0.927536,0.0167,False


In [5]:
#| label: factorial-effects
effects = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/raw/factorial_v4_2x4/"
           "factorial_effects_bca.csv"
)

# Restrict the display; the full artifact remains the authority.
effects.query(
    "family == 'unet' and metric == 'pearson' and "
    "effect_type in ['main', 'pairwise']"
)[["effect", "effect_type", "estimate", "ci_low", "ci_high"]]

,effect,effect_type,estimate,ci_low,ci_high
30,mse,main,0.245934,0.244766,0.247327
31,correlation,main,0.285433,0.283810,0.287072
32,mmd,main,0.132078,0.131069,0.133185
33,derivative,main,0.032404,0.031812,0.032986
34,mse:correlation,pairwise,-0.492726,-0.495472,-0.490522
35,mse:mmd,pairwise,-0.264514,-0.266787,-0.262463
36,mse:derivative,pairwise,-0.066770,-0.067915,-0.065569
37,correlation:mmd,pairwise,-0.257962,-0.260247,-0.255886
38,correlation:derivative,pairwise,-0.057918,-0.059192,-0.056768
39,mmd:derivative,pairwise,-0.086103,-0.087005,-0.085145


In [6]:
#| label: supplementary-seed-confirmation
#| tbl-cap: Observed validation ranges across supplementary seeds 1337 and 2026.
seed_confirmation = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/raw/factorial_v4_2x4/"
           "supplementary_seed_table.csv"
)
seed_summary = seed_confirmation.groupby(
    ["family", "slot", "mask"], as_index=False
).agg(
    seeds=("seed", lambda x: ",".join(map(str, sorted(x)))),
    validation_mse_min=("validation_mse", "min"),
    validation_mse_max=("validation_mse", "max"),
    validation_pearson_min=("validation_pearson", "min"),
    validation_pearson_max=("validation_pearson", "max"),
)
seed_summary["pearson_range"] = (
    seed_summary.validation_pearson_max -
    seed_summary.validation_pearson_min
)
seed_summary

,family,slot,mask,seeds,validation_mse_min,validation_mse_max,validation_pearson_min,validation_pearson_max,pearson_range
0,ecgaim,base,1000,"1337,2026",0.025797,0.026168,0.914991,0.915286,2.952590e-04
1,ecgaim,best,1110,"1337,2026",0.025171,0.025218,0.923050,0.923414,3.635216e-04
2,ecgaim,full,1111,"1337,2026",0.025116,0.025192,0.922865,0.923317,4.519284e-04
3,msvae,base,1000,"1337,2026",0.025941,0.025954,0.910394,0.910540,1.457356e-04
4,msvae,best,1100,"1337,2026",0.025455,0.025457,0.920692,0.920919,2.272048e-04
5,msvae,full,1111,"1337,2026",0.025478,0.025479,0.920688,0.920878,1.898016e-04
6,unet,base,1000,"1337,2026",0.028976,0.029498,0.861301,0.870364,9.062598e-03
7,unet,best,1110,"1337,2026",0.029504,0.029553,0.900516,0.904835,4.319226e-03
8,unet,full,1111,"1337,2026",0.029235,0.029544,0.903710,0.903710,6.225313e-09


In [7]:
#| label: running-mixed-level-manifest-gate
import json
import re
from itertools import product

expected_masks = {
    "1" + "".join(map(str, binary_bits)) + str(kernel)
    for binary_bits in product([0, 1], repeat=5)
    for kernel in range(5)
}
assert len(expected_masks) == 160

queue = json.loads((ROOT / "refine-logs/factorial_manifest.json").read_text())
rows = []
for phase in queue["phases"]:
    for job in phase["jobs"]:
        mask = re.search(r"--factorial_mask (\d{7})", job["cmd"]).group(1)
        seed = int(re.search(r"--seed (\d+)", job["cmd"]).group(1))
        rows.append({"phase": phase["name"], "seed": seed, "mask": mask, "job_id": job["id"]})
scheduled = pd.DataFrame(rows)

gate = []
for (phase, seed), block in scheduled.groupby(["phase", "seed"]):
    observed = set(block["mask"])
    gate.append({
        "phase": phase,
        "seed": seed,
        "jobs": len(block),
        "unique_masks": len(observed),
        "missing": len(expected_masks - observed),
        "unexpected": len(observed - expected_masks),
        "duplicate_rows": int(block["mask"].duplicated(keep=False).sum()),
    })
gate = pd.DataFrame(gate)
assert (gate[["jobs", "unique_masks"]].to_numpy() == 160).all()
assert not gate[["missing", "unexpected", "duplicate_rows"]].to_numpy().any()
gate

,phase,seed,jobs,unique_masks,missing,unexpected,duplicate_rows
0,default,42,160,160,0,0,0
1,default,200,160,160,0,0,0
2,default,201,160,160,0,0,0


In [8]:
#| label: running-mixed-level-completion-gate
#| tbl-cap: Machine-readable release gate for the still-training mixed-level factorial study.
import zipfile
import sqlite3
from datetime import datetime, timezone

STATE_PATH = ROOT / "refine-logs/queue/queue_state.json"
CHECKPOINT_CATALOG = ROOT / "results/checkpoint_store/catalog.sqlite"
live_state = json.loads(STATE_PATH.read_text())
archive_rows = {}
if CHECKPOINT_CATALOG.is_file():
    with sqlite3.connect(CHECKPOINT_CATALOG) as connection:
        connection.row_factory = sqlite3.Row
        archive_rows = {
            row["model_id"]: dict(row)
            for row in connection.execute(
                """
                SELECT model_id, size_bytes, sha256, status, asset_id,
                       asset_size_bytes, asset_digest, asset_state,
                       remote_verified_at
                FROM checkpoints
                """
            )
        }
job_rows = []
for job in live_state["jobs"]:
    checkpoint_match = re.search(
        r"--checkpoint_path\s+(\S+)", job["cmd"]
    )
    checkpoint = ROOT / checkpoint_match.group(1)
    sidecar = checkpoint.with_suffix(".metadata.json")
    sidecar_payload = None
    sidecar_error = None
    if sidecar.is_file():
        try:
            sidecar_payload = json.loads(sidecar.read_text())
        except Exception as error:
            sidecar_error = str(error)
    archive = archive_rows.get(job["id"])
    archive_verified = (
        archive is not None
        and archive.get("status") in {"remote_verified", "cached"}
        and archive.get("asset_id") is not None
        and archive.get("asset_state") == "uploaded"
        and archive.get("asset_size_bytes") == archive.get("size_bytes")
        and archive.get("asset_digest") == f"sha256:{archive.get('sha256')}"
        and archive.get("remote_verified_at") is not None
    )
    checkpoint_readable_zip = (
        checkpoint.is_file() and zipfile.is_zipfile(checkpoint)
    )
    job_rows.append({
        "job_id": job["id"],
        "status": job["status"],
        "attempts": job.get("attempts"),
        "queue_error": job.get("error"),
        "checkpoint": str(checkpoint),
        "checkpoint_present": checkpoint.is_file(),
        "checkpoint_readable_zip": checkpoint_readable_zip,
        "archive_sha256_verified": archive_verified,
        "checkpoint_recoverable": checkpoint_readable_zip or archive_verified,
        "storage_tier": (
            "local"
            if checkpoint_readable_zip
            else ("verified archive" if archive_verified else "unavailable")
        ),
        "sidecar_present": sidecar.is_file(),
        "sidecar_parseable": sidecar_payload is not None,
        "sidecar_schema_version": (
            sidecar_payload.get("schema_version")
            if sidecar_payload else np.nan
        ),
        "sidecar_identity_matches": (
            sidecar_payload is not None
            and sidecar_payload.get("run_name") == job["id"]
        ),
        "sidecar_has_split_hashes": (
            sidecar_payload is not None
            and "split_inventory_sha256" in sidecar_payload
        ),
        "sidecar_has_preprocessing": (
            sidecar_payload is not None
            and "preprocessing" in sidecar_payload
        ),
        "sidecar_error": sidecar_error,
    })
live_jobs = pd.DataFrame(job_rows)
status_counts = (
    live_jobs.status.value_counts()
    .rename_axis("queue_status").rename("jobs").reset_index()
)
print(
    "Queue snapshot:",
    datetime.fromtimestamp(
        STATE_PATH.stat().st_mtime, tz=timezone.utc
    ).isoformat(),
)
display(status_counts)

blocker_classes = pd.DataFrame([
    {
        "blocker": "queue failed or stuck",
        "jobs": int(live_jobs.status.isin(
            ["failed", "failed_other", "stuck"]
        ).sum()),
        "example_job_ids": ", ".join(
            live_jobs.loc[
                live_jobs.status.isin(["failed", "failed_other", "stuck"]),
                "job_id",
            ].head(5)
        ),
    },
    {
        "blocker": "completed but exact checkpoint unavailable",
        "jobs": int((
            live_jobs.status.eq("completed")
            & ~live_jobs.checkpoint_recoverable
        ).sum()),
        "example_job_ids": ", ".join(
            live_jobs.loc[
                live_jobs.status.eq("completed")
                & ~live_jobs.checkpoint_recoverable,
                "job_id",
            ].head(5)
        ),
    },
    {
        "blocker": "completed but metadata sidecar missing/unparseable",
        "jobs": int((
            live_jobs.status.eq("completed")
            & ~live_jobs.sidecar_parseable
        ).sum()),
        "example_job_ids": ", ".join(
            live_jobs.loc[
                live_jobs.status.eq("completed")
                & ~live_jobs.sidecar_parseable,
                "job_id",
            ].head(5)
        ),
    },
])
display(blocker_classes)

completed = live_jobs.query("status == 'completed'")
completed_artifacts_ok = (
    len(completed) == len(live_jobs)
    and completed.checkpoint_recoverable.all()
    and completed.sidecar_parseable.all()
    and completed.sidecar_identity_matches.all()
)
failed_or_stuck = live_jobs.status.isin(
    ["failed", "failed_other", "stuck", "skipped"]
).sum()
per_record_root = ROOT / "results/factorial_mixed_level/per_record"
per_record_files = (
    list(per_record_root.glob("*.parquet"))
    if per_record_root.is_dir() else []
)
source_text = (ROOT / "book/08_factorial_loss_matrix_benchmarks.qmd").read_text()
watermark_present = "PLACEHOLDER — NOT RESULTS" in source_text

release_gate = pd.DataFrame([
    {
        "gate": "1. complete manifest: 160 masks × 3 seeds",
        "observed": (
            f"{len(scheduled)} jobs; "
            f"{gate.unique_masks.min()}–{gate.unique_masks.max()} masks/seed"
        ),
        "pass": (
            len(scheduled) == 480
            and gate.missing.eq(0).all()
            and gate.unexpected.eq(0).all()
            and gate.duplicate_rows.eq(0).all()
        ),
    },
    {
        "gate": "2. all jobs completed with recoverable exact checkpoint + sidecar",
        "observed": (
            f"{len(completed)}/{len(live_jobs)} completed; "
            f"{completed.checkpoint_readable_zip.sum()} readable locally; "
            f"{completed.archive_sha256_verified.sum()} verified in archive; "
            f"{completed.sidecar_parseable.sum()} parseable sidecars"
        ),
        "pass": completed_artifacts_ok,
    },
    {
        "gate": "3. failed/partial/skipped cells explicit and resolved",
        "observed": (
            f"{failed_or_stuck} failed/stuck/skipped; "
            f"{live_jobs.status.ne('completed').sum()} not completed"
        ),
        "pass": live_jobs.status.eq("completed").all(),
    },
    {
        "gate": "4. identical record-order hashes and preprocessing",
        "observed": (
            f"{completed.sidecar_has_split_hashes.sum()}/{len(completed)} "
            "completed sidecars expose split hashes; "
            f"{completed.sidecar_has_preprocessing.sum()}/{len(completed)} "
            "expose preprocessing"
        ),
        "pass": (
            len(completed) == len(live_jobs)
            and completed.sidecar_has_split_hashes.all()
            and completed.sidecar_has_preprocessing.all()
        ),
    },
    {
        "gate": "5. per-record paired outputs for every cell",
        "observed": (
            f"{len(per_record_files)}/{len(live_jobs)} Parquet files"
        ),
        "pass": len(per_record_files) == len(live_jobs),
    },
    {
        "gate": "6. incomplete-grid refusal is active",
        "observed": "tested below with allow_partial=False",
        "pass": True,
    },
    {
        "gate": "7. placeholder watermark is visible in source",
        "observed": str(watermark_present),
        "pass": watermark_present,
    },
])

def require_releasable(report, allow_partial=False):
    blockers = report.loc[~report["pass"], "gate"].tolist()
    if blockers and not allow_partial:
        raise RuntimeError(
            "Factorial results are not releasable: " + "; ".join(blockers)
        )
    return {"releasable": not blockers, "blockers": blockers}

try:
    require_releasable(release_gate, allow_partial=False)
    refusal_test = "unexpectedly accepted"
except RuntimeError:
    refusal_test = "PASS — incomplete grid refused"
release_gate.loc[
    release_gate.gate.str.startswith("6."), "observed"
] = refusal_test
release_gate

Queue snapshot: 2026-07-31T06:00:35.427828+00:00


,queue_status,jobs
0,pending,348
1,completed,131
2,running,1


,blocker,jobs,example_job_ids
0,queue failed or stuck,0,
1,completed but exact checkpoint unavailable,0,
2,completed but metadata sidecar missing/unparse...,49,"f_1000001_s42, f_1000003_s42, f_1000004_s42, f..."


,gate,observed,pass
0,1. complete manifest: 160 masks × 3 seeds,480 jobs; 160–160 masks/seed,True
1,2. all jobs completed with recoverable exact c...,131/480 completed; 102 readable locally; 29 ve...,False
2,3. failed/partial/skipped cells explicit and r...,0 failed/stuck/skipped; 349 not completed,False
3,4. identical record-order hashes and preproces...,21/131 completed sidecars expose split hashes;...,False
4,5. per-record paired outputs for every cell,0/480 Parquet files,False
5,6. incomplete-grid refusal is active,PASS — incomplete grid refused,True
6,7. placeholder watermark is visible in source,True,True


In [9]:
#| label: checkpoint-store-live-summary
#| tbl-cap: Live exact-checkpoint storage tiers at render time.
if CHECKPOINT_CATALOG.is_file():
    with sqlite3.connect(CHECKPOINT_CATALOG) as connection:
        checkpoint_storage = pd.read_sql_query(
            """
            SELECT status AS storage_status,
                   COUNT(*) AS models,
                   ROUND(SUM(size_bytes) / 1073741824.0, 3) AS logical_GiB,
                   ROUND(SUM(CASE WHEN local_path IS NOT NULL
                                  THEN size_bytes ELSE 0 END)
                         / 1073741824.0, 3) AS cataloged_local_GiB
            FROM checkpoints
            GROUP BY status
            ORDER BY status
            """,
            connection,
        )
else:
    checkpoint_storage = pd.DataFrame([{
        "storage_status": "catalog unavailable",
        "models": 0,
        "logical_GiB": 0.0,
        "cataloged_local_GiB": 0.0,
    }])
checkpoint_storage

,storage_status,models,logical_GiB,cataloged_local_GiB
0,local,101,6.289,6.289
1,remote_verified,29,1.751,0.000
2,uploading,1,0.076,0.076


In [10]:
#| label: planned-mixed-level-placeholder
#| eval: false
rng = np.random.default_rng(20260731)
placeholder = pd.DataFrame(sorted(expected_masks), columns=["mask"])
placeholder["PLACEHOLDER_metric"] = rng.normal(size=len(placeholder))
placeholder["status"] = "PLACEHOLDER — NOT RESULTS"
placeholder.head()